In [42]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
import torch
import pandas as pd



In [43]:

# ==============================
# 1. LOAD DATA
# ==============================
df_train = pd.read_csv("/content/data/clean_train.csv")
df_val   = pd.read_csv("/content/data/clean_val.csv")


In [44]:

# ==============================
# 2. TOKENIZER + MODEL
# ==============================
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=3,
    dropout=0.3,
    attention_dropout=0.3
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [45]:

# ==============================
# 3. ENCODING
# ==============================
def encode_texts(texts):
    return tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=64
    )

train_encodings = encode_texts(df_train['tweet_content'].tolist())
val_encodings   = encode_texts(df_val['tweet_content'].tolist())

train_labels = torch.tensor(df_train['label'].values)
val_labels   = torch.tensor(df_val['label'].values)


In [46]:

# ==============================
# 4. DATASET CLASS
# ==============================
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = SentimentDataset(train_encodings, train_labels)
val_dataset   = SentimentDataset(val_encodings, val_labels)


In [47]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# ==============================
# 5. TRAINING ARGUMENTS
# ==============================
training_args = TrainingArguments(
    output_dir='/content/models/distilbert_model',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    fp16=True,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss'
)

In [48]:

# ==============================
# 6. TRAINER
# ==============================
from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [49]:

# ==============================
# 7. TRAIN + EVALUATE
# ==============================
trainer.train()
trainer.evaluate()


Epoch,Training Loss,Validation Loss
1,0.637758,0.469735
2,0.472324,0.263798
3,0.374495,0.141385
4,0.320935,0.096726
5,0.218003,0.079842


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'eval_loss': 0.07984215021133423,
 'eval_runtime': 0.515,
 'eval_samples_per_second': 1605.892,
 'eval_steps_per_second': 100.975,
 'epoch': 5.0}

In [50]:

# ==============================
# 8. SAVE MODEL
# ==============================
model.save_pretrained("/content/models/distilbert_model")
tokenizer.save_pretrained("/content/models/distilbert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/models/distilbert_model/tokenizer_config.json',
 '/content/models/distilbert_model/tokenizer.json')